# 08a - Density, Distributions, and Evidence for Clusters

K-means groups observations around centers. Agglomerative hierarchical clustering joins observations and clusters under a linkage rule, giving us several resolutions to inspect. Both have helped us ask whether customer profiles could guide a service team's investigation.

What looks like a group in a plot is a starting point. The features, their scales, and the relationships we want to capture affect what should count as a group. Today we will examine density-connected groups and overlapping probability distributions, then ask what evidence would support using a proposed clustering.

## References

- [Scikit-learn DBSCAN guide](https://scikit-learn.org/stable/modules/clustering.html#dbscan) and [DBSCAN API](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html): neighborhoods, point roles, parameters, and fitted outputs.
- [Python Data Science Handbook: Gaussian Mixtures](https://jakevdp.github.io/PythonDataScienceHandbook/05.12-gaussian-mixtures.html): geometric and probabilistic explanation. Its older code uses the retired `GMM` class; this notebook uses `GaussianMixture`.
- [Scikit-learn Gaussian mixture guide](https://scikit-learn.org/stable/modules/mixture.html) and [GaussianMixture API](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html): fitting, component assumptions, and membership probabilities.
- [Gap statistic reference and implementation notes](https://stat.ethz.ch/R-manual/R-devel/RHOME/library/cluster/html/clusGap.html): reference distributions and cluster-count selection.
- [Gao, Bien, and Witten: Selective Inference for Hierarchical Clustering](https://arxiv.org/abs/2012.02936): why ordinary tests after clustering can be misleading.
- [Huang et al.: Statistical Significance of Clustering Using Soft Thresholding](https://arxiv.org/abs/1305.5879): formal single-Gaussian cluster testing and its calibration challenges. The classroom simulation below illustrates reference-model reasoning; it is not an implementation of SigClust.

## Python Setup

Run the cells in order. The examples are generated here, using two coordinates measured in the same arbitrary unit. No data download is needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Circle, Ellipse
from scipy.spatial.distance import cdist
from scipy.stats import ttest_ind
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

# Bound parallel work so repeated small fits remain responsive.
thread_limit = threadpool_limits(limits=2)
COLORS = ["#4477AA", "#EE6677", "#228833", "#AA3377"]
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})


def plot_groups(ax, x, labels, title):
    for label in np.unique(labels):
        selected = labels == label
        ax.scatter(
            *x[selected].T, s=22, alpha=0.8,
            color="0.25" if label == -1 else COLORS[int(label) % len(COLORS)],
            marker="x" if label == -1 else "o",
        )
    ax.set(title=title, xlabel="Coordinate 1", ylabel="Coordinate 2")
    ax.set_aspect("equal", adjustable="box")


def add_component_contours(ax, model):
    for component, (mean, covariance) in enumerate(
        zip(model.means_, model.covariances_, strict=True)
    ):
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
        ax.add_patch(Ellipse(
            mean, 4 * np.sqrt(eigenvalues[1]), 4 * np.sqrt(eigenvalues[0]),
            angle=angle, fill=False, edgecolor=COLORS[component], linewidth=2,
        ))
        ax.scatter(*mean, marker="X", s=95, color=COLORS[component], edgecolor="black")


def dbscan_summary(model):
    return {
        "Clusters": len(set(model.labels_) - {-1}),
        "Noise observations": int(np.sum(model.labels_ == -1)),
        "Core observations": len(model.core_sample_indices_),
    }

## Density-Based Clustering

Suppose observations trace two curved regions. Assigning every point to its nearest center can cut across those regions. Another definition of a cluster is a region we can travel through by moving between sufficiently dense neighborhoods.

**DBSCAN** stands for Density-Based Spatial Clustering of Applications with Noise. It forms groups using this neighborhood idea and can leave some observations unassigned. We choose the local density rule; the resulting number of clusters is an output.

In [ ]:
curves, _ = make_moons(n_samples=300, noise=0.06, random_state=7130)
isolated = np.array([[-1.5, 1.4], [2.5, 1.4], [-1.5, -0.8], [2.5, -0.8]])
curves = np.vstack([curves, isolated])

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.scatter(*curves.T, color="0.35", s=22)
ax.set(title="Constructed Curved Regions and Four Isolated Observations",
       xlabel="Coordinate 1", ylabel="Coordinate 2")
ax.set_aspect("equal")
plt.show()

The two arcs look like groups. How could an algorithm follow each curve without joining it to the other one? DBSCAN makes that decision locally, by counting nearby points.

### Neighborhoods, Core Points, and Border Points

Two settings define the neighborhood rule:

- `eps` (epsilon, $\epsilon$) is the radius within which observations count as neighbors, including points exactly at the radius.
- `min_samples` is the number of observations required in that neighborhood for a point to be **core**. The point itself counts.

A **border** point is not core but lies within the radius of a core point. A **noise** point is neither core nor a neighbor of any core point. These roles describe the current data under the selected settings.

Start with seven points and use a radius of 0.5 and a minimum count of four. Every point lies on the quarter-unit grid, so the radius spans two grid steps. The dashed blue circle is D's neighborhood; the shaded circle is E's. Count points inside or on each circle, including its center, and compare with the table.

In [ ]:
small_points = np.array([
    [-0.25, 0.0], [0.0, 0.25], [0.0, 0.0], [0.25, 0.0],
    [0.75, 0.0], [1.25, 0.0], [2.0, 1.0],
])
point_names = list("ABCDEFG")
neighbor_counts = (cdist(small_points, small_points) <= 0.5).sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
ax.add_patch(Circle(small_points[4], 0.5, facecolor=COLORS[1], alpha=0.10,
                    edgecolor="none", zorder=1))
for row, color, style, label in [
    (3, COLORS[0], "--", "D's neighborhood: 5 points"),
    (4, COLORS[1], "-", "E's neighborhood: 3 points"),
]:
    ax.add_patch(Circle(small_points[row], 0.5, fill=False, linestyle=style,
                        edgecolor=color, linewidth=2, label=label, zorder=2))
ax.scatter(*small_points.T, color="0.2", s=70, zorder=3)
for name, xy in zip(point_names, small_points, strict=True):
    offset = (0, -19) if name == "C" else (0, 10)
    ax.annotate(name, xy, xytext=offset, textcoords="offset points", ha="center",
                fontweight="bold", zorder=4)
ax.set(xlim=(-0.5, 2.25), ylim=(-0.65, 1.3), xlabel="Coordinate 1", ylabel="Coordinate 2",
       xticks=np.arange(-0.5, 2.26, 0.25), yticks=np.arange(-0.5, 1.26, 0.25),
       title="Neighborhoods of D and E: Radius 0.5 (Two Grid Steps)")
ax.xaxis.set_major_formatter("{x:g}")
ax.yaxis.set_major_formatter("{x:g}")
ax.set_axisbelow(True)
ax.grid(color="0.88", linewidth=0.7)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=10)
plt.show()
display(pd.DataFrame({
    "Point": point_names, "Neighbors including self": neighbor_counts,
    "Core with minimum count 4": neighbor_counts >= 4,
}).set_index("Point"))

A, B, C, and D have enough neighbors to be core. E's neighborhood contains exactly D, E, and F. D and F lie on its boundary and both count. E has only three neighbors, including itself, but is close to core point D, so E is border. F is close to E but not to any core point. F and G are noise.

DBSCAN builds a cluster by following connections among core points, then including adjacent border points. A border point can join a cluster but cannot extend it to its own neighbors. This is how a chain of core neighborhoods can follow a curved region. Points at opposite ends can be much farther apart than `eps`.

### Pause: A Border Is Not a Bridge

E belongs to the cluster containing D. F is close to E, so why does F remain noise? With a radius of 0.5 and `min_samples=4`, check whether each point is near a **core point**, rather than just any cluster member.

##### Answer

D is a core point because its neighborhood contains five points, meeting the minimum count of four. E lies within D's radius, so E joins the cluster containing D.

E's own neighborhood contains only D, E, and F: three points, including E itself. E is therefore a border point, not a core point.

F lies within E's radius, but outside the radius of every core point. Being close to border point E is not enough, so F remains noise. **A non-core point joins a cluster only if it is within the radius of a core point.**

If we lowered `min_samples` to three while keeping the same radius, E would become core. F would then join the cluster as a border point because it is within E's radius.

### Fit DBSCAN and Read the Result

For the larger curved example, try `min_samples=5` and `eps=0.20`. We will vary the radius to see what these choices do; another dataset would need its own neighborhood settings.

`fit()` learns the point roles and memberships. The fitted object's `labels_` array has one label per input row. Nonnegative integers name clusters; `-1` marks noise. `core_sample_indices_` identifies the rows that are core.

In [ ]:
density_model = DBSCAN(eps=0.20, min_samples=5).fit(curves)
density_labels = density_model.labels_

display(pd.Series(dbscan_summary(density_model), name="DBSCAN result"))
display(pd.Series(density_labels, name="Label").value_counts().sort_index().rename("Rows"))

In [ ]:
center_model = KMeans(n_clusters=2, n_init=20, random_state=7130).fit(curves)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True, layout="constrained")
plot_groups(axes[0], curves, center_model.labels_, "K-Means: Two Nearest-Center Groups")
plot_groups(axes[1], curves, density_labels, "DBSCAN: Two Connected Groups; Crosses Are Noise")
plt.show()

DBSCAN follows the two arcs and leaves the four isolated observations as noise. K-means assigns every observation, splitting the arcs according to proximity to its centers. Both results follow their definitions of a group. Their labels are arbitrary names, so compare membership patterns rather than matching colors across methods.

A noise point has no cluster under these settings. Before removing it, investigate: is it an error, a rare but legitimate case, or a point excluded by our radius and minimum count?

### Neighborhood Choices

Hold the observations, distance, and minimum count fixed. Change only the radius. Before looking at the result, consider what a larger neighborhood could do to connections between the arcs.

In [ ]:
radius_models = {
    radius: DBSCAN(eps=radius, min_samples=5).fit(curves)
    for radius in [0.12, 0.20, 0.35]
}
radius_summary = pd.DataFrame({
    radius: dbscan_summary(model) for radius, model in radius_models.items()
}).T
radius_summary.index.name = "Radius"
display(radius_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharex=True, sharey=True, layout="constrained")
for ax, (radius, model) in zip(axes, radius_models.items(), strict=True):
    plot_groups(ax, curves, model.labels_, f"Radius {radius:.2f}")
plt.show()

At 0.12 one arc fragments, giving three clusters and six noise points. At 0.20 the two arcs remain distinct. At 0.35 connections join them into one cluster. Increasing the radius can therefore change the interpretation, even when the number of noise points barely changes.

At a fixed radius, increasing `min_samples` makes becoming core harder. This creates a problem when one group is dense and another sparse: settings that connect the sparse group may also join regions we wanted to keep separate.

Scaling changes which points fall inside the radius. As with K-means, choose the features and their scales before interpreting the distances. An `eps` of 0.20 has meaning only in those coordinates.

Here, radii 0.20 and 0.25 give the same memberships. Finding a range like this is more reassuring than finding one setting that produces an attractive picture.

## Gaussian Mixture Models

Density connectivity is one way to define a group. Another possibility is that observations come from several overlapping distributions. Their locations and spreads may differ, and an observation in an overlap may be compatible with more than one component.

A **Gaussian mixture model**, or GMM, describes the data with a weighted combination of Gaussian distributions. A Gaussian is the familiar normal distribution: its mean locates the center and its variance describes spread. With two features, a Gaussian describes a cloud whose spread can differ by direction.

In [ ]:
cloud_rng = np.random.default_rng(7130)
clouds = np.vstack([
    cloud_rng.multivariate_normal([-1.4, 0], [[1.2, 1.0], [1.0, 1.2]], 240),
    cloud_rng.multivariate_normal([1.4, 0], [[0.7, 0.5], [0.5, 0.8]], 160),
])
fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
ax.scatter(*clouds.T, color="0.35", s=20, alpha=0.75)
ax.set(title="400 Observations From Two Overlapping Distributions",
       xlabel="Coordinate 1", ylabel="Coordinate 2")
ax.set_aspect("equal")
plt.show()

We generated 240 observations from one tilted distribution and 160 from another. The fitting methods receive only their coordinates. Can they recover the overlapping structure without being told which distribution generated each point?

### Centers, Spreads, and Component Weights

Each Gaussian component has three kinds of fitted quantities:

| Quantity | Meaning |
| --- | --- |
| Mean | The center in the feature space |
| Covariance | Variation along each feature and how features vary together |
| Weight | The component's share of the fitted mixture; the weights sum to one |

In two dimensions, covariance controls the width and tilt of elliptical contours. A tilted cloud means that deviations in the two features tend to move together. Allowing a separate covariance for each component gives the model more flexibility than comparing distance to centers alone.

### How the Mixture Is Fitted

The component distributions are unknown, and so are the observations' memberships. Fitting alternates between estimates of the two:

1. Start with provisional component means, covariances, and weights.
2. Use those components to estimate relative membership probabilities for every observation.
3. Update the components using all observations, weighted by those membership probabilities.
4. Repeat until the improvement in likelihood is small enough or an iteration limit is reached.

This is **expectation-maximization**, or EM. Likelihood measures how well the fitted distributions account for the observed data. Membership estimates guide the next component estimates, which then give us revised memberships. The data and requested component count stay fixed. Different starts can lead to different fits, so we try several.

### Fit and Inspect the Components

Start with two components. `covariance_type="full"` lets each have its own spread and orientation. As with K-means, `n_init=10` means ten complete fits from different starts. Here the model retains the fit with the highest likelihood. `random_state` makes the starts reproducible.

After fitting, check `converged_`. The model stores the learned means, covariances, and weights. `predict(X)` returns the most probable component for each row, while `predict_proba(X)` retains all the membership probabilities.

In [ ]:
mixture = GaussianMixture(
    n_components=2, covariance_type="full", n_init=10, random_state=7130,
).fit(clouds)
mixture_labels = mixture.predict(clouds)
membership = mixture.predict_proba(clouds)

print(f"Converged: {mixture.converged_}")
component_summary = pd.DataFrame({
    "Mean coordinate 1": mixture.means_[:, 0],
    "Mean coordinate 2": mixture.means_[:, 1],
    "Mixture weight": mixture.weights_,
    "Hard-label count": np.bincount(mixture_labels, minlength=2),
}).rename_axis("Component")
display(component_summary.round(3))

In [ ]:
cloud_kmeans = KMeans(n_clusters=2, n_init=20, random_state=7130).fit(clouds)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True, layout="constrained")
plot_groups(axes[0], clouds, cloud_kmeans.labels_, "K-Means: Nearest Center")
plot_groups(axes[1], clouds, mixture_labels, "GMM: Most Probable Component")
add_component_contours(axes[1], mixture)
plt.show()

The fitted mixture follows the tilted distributions more closely, while K-means cuts across part of the longer cloud. The ellipses show equal-density contours within each Gaussian component. They show spread, not hard membership boundaries or confidence intervals for the means. Gaussian density continues beyond each ellipse.

The fitted weights are about 0.583 and 0.417. They describe the mixture, whereas the hard-label counts are 236 and 164. Weighted membership and assigning each row entirely to one component do not produce exactly the same proportions.

### Soft Membership

A hard label hides how closely the competing components explain a point. Inspect three observations: one near each fitted mean and one where the membership probabilities are closest to 50/50. Each row below belongs to one observation; each probability column belongs to one component.

In [ ]:
selected_rows = [
    int(np.argmin(np.linalg.norm(clouds - mixture.means_[0], axis=1))),
    int(np.argmin(np.abs(membership[:, 0] - 0.5))),
    int(np.argmin(np.linalg.norm(clouds - mixture.means_[1], axis=1))),
]
selected_membership = pd.DataFrame({
    "Example": ["A", "B", "C"],
    "Observation row": selected_rows,
    "Coordinate 1": clouds[selected_rows, 0],
    "Coordinate 2": clouds[selected_rows, 1],
    "P(component 0)": membership[selected_rows, 0],
    "P(component 1)": membership[selected_rows, 1],
    "Hard label": mixture_labels[selected_rows],
}).set_index("Example")
display(selected_membership.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5), layout="constrained")
points = ax.scatter(*clouds.T, c=membership[:, 1], cmap="coolwarm", vmin=0, vmax=1, s=22)
for name, row in zip("ABC", selected_rows, strict=True):
    ax.annotate(name, clouds[row], xytext=(6, 8), textcoords="offset points",
                bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85})
ax.set(title="Relative Membership in Component 1", xlabel="Coordinate 1", ylabel="Coordinate 2")
ax.set_aspect("equal")
fig.colorbar(points, ax=ax, label="P(component 1)", shrink=0.8)
plt.show()

Observation B receives probabilities about 0.516 and 0.484. Component 0 wins the hard label by a narrow margin. Keeping both probabilities shows how ambiguous that assignment is under this model.

GMM calculates membership probabilities by comparing the components' weighted densities at a point and normalizing those contributions to sum to one. These probabilities describe relative membership. Even when every component assigns very low density to a point, one can receive most of its membership probability. A confident assignment therefore does not necessarily mean the observation is typical.

### Pause: What Does the Label Lose?

A and B receive the same hard label. Which assignment would you be more willing to act on? What would you do with B?

##### Answer

A is strongly associated with one component under the model; B is nearly evenly divided. We might leave B unassigned or review it before acting. The choice depends on the cost of a wrong assignment. Taking the largest probability automatically would hide this decision.

### The Component Count Is Still a Choice

We supplied the count of two. Now compare one, two, and three components with the same covariance and fitting settings. What does each added component help describe?

In [ ]:
component_models = {
    count: GaussianMixture(
        n_components=count, covariance_type="full", n_init=10, random_state=7130,
    ).fit(clouds)
    for count in [1, 2, 3]
}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True, layout="constrained")
for ax, (count, model) in zip(axes, component_models.items(), strict=True):
    plot_groups(ax, clouds, model.predict(clouds), f"{count} Component{'s' if count > 1 else ''}")
    add_component_contours(ax, model)
plt.show()

One component describes overall variation. Two distinguish the overlapping clouds. Three split the longer cloud further. A component can help approximate the shape of a distribution without identifying another population. Before interpreting it as a group, inspect the observations it describes and ask what that distinction would mean.

Unlike DBSCAN's fixed radius, full covariance lets the model estimate different spreads along different directions. This accommodates elongated groups, but each component still has a Gaussian shape. Check whether those shapes fit the data; more components may be compensating for a poor shape assumption.

### Choosing the Covariance Type

We used `covariance_type="full"` so each component could follow its own tilted cloud. The alternatives restrict which shapes the model can fit.

| `covariance_type` | Component Shapes in Two Dimensions | When It May Be Useful |
| --- | --- | --- |
| `"full"` | Each component has its own size, elongation, and tilt. | Groups have different spreads and correlations, with enough observations to estimate them. |
| `"tied"` | All components share the same size, elongation, and tilt; their centers differ. | Groups have similar internal variation. Sharing covariance reduces the amount to estimate. |
| `"diag"` | Each component has its own horizontal and vertical spread, with no tilt. | Within-component feature correlations are negligible, or estimating all correlations requires too much data. |
| `"spherical"` | Each component is circular, with its own radius. | Roughly round groups are plausible. Feature scaling matters because spread is equal in every direction within a component. |

These restrictions trade flexibility for fewer parameters. With ten features, a full covariance needs 55 values per component, diagonal needs ten, and spherical needs one. Tied estimates one shared set of 55 covariance values.

**Spherical GMM is closest in spirit to K-means:** both favor round groups, but GMM supplies membership probabilities. Its components can also have different spreads and weights, so their assignments need not match K-means. The probabilities come from fitting the mixture model, rather than being added to a K-means partition.

Compare the same observations with two components in every panel. Only the covariance type changes; all fits use ten starts and the same random seed. Which shapes can follow the tilt, and which assignments change when that flexibility is removed?

In [ ]:
covariance_models = {
    kind: GaussianMixture(
        n_components=2, covariance_type=kind, n_init=10, random_state=7130,
    ).fit(clouds)
    for kind in ["full", "tied", "diag", "spherical"]
}
if not all(model.converged_ for model in covariance_models.values()):
    raise RuntimeError("A covariance comparison fit did not converge.")

In [ ]:
fig, axes = plt.subplots(
    1, 4, figsize=(16, 4.4), sharex=True, sharey=True, layout="constrained",
)
shape_notes = {
    "full": "Separate tilted ellipses",
    "tied": "One shared covariance",
    "diag": "Axis-aligned ellipses",
    "spherical": "Circles of different sizes",
}
for ax, (kind, model) in zip(axes, covariance_models.items(), strict=True):
    # Order colors by the horizontal means; component IDs themselves are arbitrary.
    order = np.argsort(model.means_[:, 0])
    raw_labels = model.predict(clouds)
    labels = np.argsort(order)[raw_labels]
    ax.scatter(*clouds.T, c=[COLORS[label] for label in labels], s=14, alpha=0.75)
    for color_index, component in enumerate(order):
        if kind == "full":
            covariance_matrix = model.covariances_[component]
        elif kind == "tied":
            covariance_matrix = model.covariances_
        elif kind == "diag":
            covariance_matrix = np.diag(model.covariances_[component])
        else:
            covariance_matrix = np.eye(2) * model.covariances_[component]
        values, vectors = np.linalg.eigh(covariance_matrix)
        angle = np.degrees(np.arctan2(vectors[1, 1], vectors[0, 1]))
        ax.add_patch(Ellipse(
            model.means_[component], 4 * np.sqrt(values[1]), 4 * np.sqrt(values[0]),
            angle=angle, fill=False, edgecolor=COLORS[color_index], linewidth=2,
        ))
        ax.scatter(*model.means_[component], marker="X", s=65,
                   color=COLORS[color_index], edgecolor="black", zorder=4)
    ax.set(title=f"{kind}: {shape_notes[kind]}", xlabel="Coordinate 1",
           xlim=(clouds[:, 0].min() - 0.5, clouds[:, 0].max() + 0.5),
           ylim=(clouds[:, 1].min() - 0.5, clouds[:, 1].max() + 0.5))
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.15)
axes[0].set_ylabel("Coordinate 2")
fig.suptitle("Same Overlapping Clouds and Two Components: Changing Covariance Type")
plt.show()

Full covariance lets each ellipse tilt and stretch separately. Tied covariance retains tilt but shares the same shape and size. Diagonal covariance removes tilt; spherical covariance also removes elongation. Refitting under those restrictions changes the means and some assignments as well.

Colors show the most probable component, ordered by horizontal mean within each fit. X marks the means. The contours use the same two-standard-deviation radius along each component's principal axes; they show component spread, not hard membership boundaries.

## Four Ways to Describe Groups

Use this table to explain a result that initially seems odd. If K-means splits an arc, inspect its nearest-center rule. If DBSCAN leaves a point unassigned, inspect its neighborhood. If a GMM label seems too definite, inspect its membership probabilities.

| Question | K-Means | Agglomerative Hierarchical | DBSCAN | Gaussian Mixture |
| --- | --- | --- | --- | --- |
| What joins observations? | Proximity to a center under squared distance | Successive merges under a linkage rule | Connections among dense neighborhoods | Association with a fitted component distribution |
| What structure does it favor or expose? | Compact nearest-center partitions | Nested groupings; geometry depends on linkage | Irregular connected regions | Overlapping distributions with elliptical component contours |
| Which choices matter? | Representation, cluster count, starts | Representation, distance, linkage, cut | Representation, distance, radius, minimum count | Representation, component count, covariance assumptions, starts |
| What distinctive output appears? | Centers and hard labels | Merge history and alternative cuts | Core/border roles and noise | Component parameters and soft membership |
| Is every observation assigned? | Yes | Yes, for an ordinary full-tree cut | No | Probabilities cover all components; hard labels can be requested |
| What about new observations? | Nearest fitted center | Requires a separately chosen rule or refitting | No built-in prediction rule in this estimator | Membership under the fitted distributions |
| What deserves investigation? | Forced splits, competing fits, outlier effects | Linkage or cut sensitivity, chaining | Excess noise, merged regions, varying density | Shape mismatch, ambiguous memberships, unstable or excessive components |

### Comparison Checkpoint

In the clustering microscope, compare all four methods on the customers and the larger dataset. Keep the features and scales fixed. Choose a point whose assignment changes between two methods: which rule explains the change? Use the geometric cases if the reason is hard to see.

Then change one setting, such as DBSCAN's radius or the requested component count. Which grouping would you keep for further investigation, and what evidence would help you choose?

## Evidence for a Clustering

The microscope gives us several plausible results. How should we choose among them? A score, a stable partition, and a useful service distinction answer different parts of that question.

| Question | Evidence to inspect |
| --- | --- |
| Does the grouping capture the structure we care about? | Members, profiles, and geometry |
| What does another cluster add? | Alternative counts and the distinctions they introduce |
| Does the grouping persist? | Changes in settings, features, starts, or sampled observations |
| Could one population produce separation like this? | Simulations from a justified population model |
| Would the grouping help us act? | Relevant external measurements, another period, and domain knowledge |

### Elbow, Silhouette, and Gap

The elbow looks for diminishing reductions in **inertia**, the same sum of squared distances from observations to their assigned centers used earlier. Silhouette asks whether points are closer to their own cluster than to the nearest competing cluster. It is undefined for one cluster because there is no competitor.

A higher silhouette score indicates better separation by this measure. When several counts give nearly the same score, **parsimony** favors a simpler grouping unless the extra clusters add a useful distinction. This is an analytical judgment alongside the score, not a formal silhouette rule. Silhouette also favors compact, separated groups and can undervalue a useful curved grouping.

Adding more centers reduces inertia even in random data. The **gap statistic** asks whether another cluster is doing meaningful work: **does it improve the fit beyond what we would expect from partitioning random reference data?**

We return to the same 72 simulated customers used in 07a, standardizing requests and average minutes per request as before. For comparison, we generate 100 reference datasets of 72 points each, spread uniformly across a region with a similar range and orientation. This is our **no-group reference**: it has no deliberately planted groups, although random clumps can occur. It covers a similar region without reproducing the customer data's exact variance or density.

We fit K-means to the customers and each reference dataset with one through eight clusters. Gap compares their inertias on a logarithmic scale. A larger gap means the customer clustering is more compact relative to these no-group reference fits.

Starting with one cluster, ask whether adding another increases that advantage by more than the variation in the simulations. If it does, continue. If the increase is small enough to fall within the simulation allowance, or gap decreases, stop and keep the smaller count. The bars show that allowance. **Choose the simpler grouping once another cluster no longer gives a clear improvement relative to the random reference.**

Here, “meaningful work” means describing structure relative to this uniform reference. Whether another group would help the service team make a decision still needs separate evidence.

In [ ]:
def compare_cluster_counts(x, max_k=8, n_reference=100, seed=810):
    """Compare K-means counts with a PCA-aligned uniform gap reference.

    Implementation detail: center and rotate into principal-component directions,
    keeping both dimensions and their scales. Draw uniformly within those ranges.
    Rotation preserves observed distances but determines the reference box.

    Gap(k) = mean_b(log W_reference[b, k]) - log W_observed[k].
    Average the logs, not the inertias before taking a log.
    s(k) = SD_b(log W_reference[b, k]) * sqrt(1 + 1/B).
    Choose the first k with Gap(k) >= Gap(k+1) - s(k+1).
    Bars display s(k), not confidence intervals for a true count. The last
    candidate supplies the next-count comparison; extend the range if none passes.

    """
    counts = np.arange(1, max_k + 1)
    fits = [KMeans(n_clusters=int(k), n_init=20, random_state=7130).fit(x)
            for k in counts]
    inertia = np.array([fit.inertia_ for fit in fits])
    silhouette = [np.nan] + [silhouette_score(x, fit.labels_) for fit in fits[1:]]

    centered = x - x.mean(axis=0)
    _, _, directions = np.linalg.svd(centered, full_matrices=False)
    rotated = centered @ directions.T
    rng = np.random.default_rng(seed)
    reference_log_w = np.empty((n_reference, max_k))
    for b in range(n_reference):
        reference = rng.uniform(rotated.min(axis=0), rotated.max(axis=0), size=x.shape)
        for j, k in enumerate(counts):
            fit = KMeans(n_clusters=int(k), n_init=20, random_state=7130).fit(reference)
            reference_log_w[b, j] = np.log(fit.inertia_)

    gap = reference_log_w.mean(axis=0) - np.log(inertia)
    allowance = reference_log_w.std(axis=0, ddof=1) * np.sqrt(1 + 1 / n_reference)
    qualifies = gap[:-1] >= gap[1:] - allowance[1:]
    selected = int(counts[:-1][qualifies][0]) if qualifies.any() else None
    table = pd.DataFrame({
        "k": counts, "Inertia": inertia, "Silhouette": silhouette,
        "Mean reference log W": reference_log_w.mean(axis=0),
        "Gap": gap, "Simulation allowance": allowance,
    }).set_index("k")
    return table, selected

The helper uses the same 20-start K-means procedure for observed and reference data. Compare the three curves below: where does inertia bend, where is silhouette highest, and where does the gap rule first stop?

In [ ]:
customer_rng = np.random.default_rng(7130)
customers = pd.DataFrame({
    "requests": customer_rng.integers(1, 13, size=72),
    "avg_minutes": customer_rng.integers(8, 65, size=72),
}, index=pd.Index(range(1, 73), name="customer"))
scaled_customers = StandardScaler().fit_transform(customers)

count_comparison, gap_choice = compare_cluster_counts(scaled_customers, max_k=8)
silhouette_choice = int(count_comparison["Silhouette"].idxmax())
display(count_comparison.round(3))
print(f"Largest silhouette among tested counts: k = {silhouette_choice}")
print(f"Gap rule: k = {gap_choice}" if gap_choice is not None
      else "Gap rule: no qualifying count yet; extend the range.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout="constrained")
counts = count_comparison.index.to_numpy()
axes[0].plot(counts, count_comparison["Inertia"], "o-")
axes[0].set(title="Elbow: Diminishing Returns", ylabel="Inertia (squared-distance sum)")
axes[1].plot(counts, count_comparison["Silhouette"], "o-")
axes[1].axvline(silhouette_choice, color=COLORS[1], linestyle="--",
               label=f"Highest tested: k = {silhouette_choice}")
axes[1].set(title="Silhouette: Larger Is Better", ylabel="Mean silhouette")
axes[1].legend()
axes[2].errorbar(counts, count_comparison["Gap"],
                 yerr=count_comparison["Simulation allowance"], fmt="o-", capsize=4)
if gap_choice is not None:
    axes[2].axvline(gap_choice, color=COLORS[1], linestyle="--",
                   label=f"Rule selects k = {gap_choice}")
    axes[2].legend()
axes[2].set(title="Gap: Apply the Selection Rule", ylabel="Gap with simulation allowance")
for ax in axes:
    ax.set(xlabel="Number of K-means clusters (k)", xticks=counts)
    ax.grid(alpha=0.2)
fig.suptitle("72 Standardized Customers, Same K-Means Fits, Three Diagnostics")
plt.show()

Silhouette is highest at the largest count we tested. How different is that score from the scores at three through seven clusters? Would you call eight a clear choice?

Inertia falls from 144 at one cluster to about 91 at two and 54 at three; later reductions are smaller. As in 07a, three is worth inspecting as a possible elbow.

Silhouette improves substantially through three clusters, rises a little at four, then fluctuates with little further gain. Eight has the highest tested score (0.419), but scores from three through eight are close, about 0.40–0.42. Parsimony makes three or four reasonable candidates to inspect before accepting eight: do the additional groups reveal distinctions useful to the service team?

Gap selects one. Adding a second center improves the uniform-reference fits proportionally more than the customer fit, so gap falls and the rule stops. These independently generated customer measurements contain no planted types, although partitions can still help us ask about differences in service needs.

The diagnostics leave us with different questions: what does another group add, how well separated is it, and how does the fit compare with the chosen reference? Inspect the customer profiles before choosing a grouping for the service team.

**Gap helps choose a count under a particular reference. It is not a p-value or proof of the true number of groups.**

### Stability Under Changed Observations

Would we find similar groups if our sample had included a somewhat different set of observations? To explore that question, treat the observed dataset as a stand-in for the population. Repeatedly draw a random 80% of its rows without replacement, fit the two-component GMM again, and compare the assignments for observations included in both fits. The 80% fraction is a demonstration choice.

Changing random starts on the full dataset checks sensitivity to initialization. Refitting subsets explores sensitivity to which observations are included. Each refit here uses ten starts to reduce the influence of initialization while keeping the component count and covariance form fixed.

These subsamples overlap and reuse observations we already have. They show how the grouping responds to changes within this dataset; they do not reproduce collecting independent new samples from the actual population.

Compare each new partition with the original fit on the sampled rows. The **adjusted Rand index**, or ARI, checks whether pairs of observations stay together or apart, allowing for chance agreement. It ignores label numbers. One means identical partitions; zero means chance-level agreement, and negative values mean less agreement than expected by chance.

In [ ]:
stability_rng = np.random.default_rng(809)
stability_scores = []
for repeat in range(30):
    rows = stability_rng.choice(len(clouds), size=int(0.8 * len(clouds)), replace=False)
    refit = GaussianMixture(
        n_components=2, covariance_type="full", n_init=10, random_state=repeat,
    ).fit(clouds[rows])
    if not refit.converged_:
        raise RuntimeError("A subsample fit did not converge; inspect before summarizing.")
    stability_scores.append(
        adjusted_rand_score(mixture_labels[rows], refit.predict(clouds[rows]))
    )

display(pd.Series(stability_scores, name="Subsample agreement (ARI)").describe().round(3))

The median ARI is about 0.99 and the minimum about 0.95: most pairs keep their relationship when some rows are omitted. The grouping is stable under this check. Another period of observations would check whether the grouping recurs in new data, beyond the sample we have reused here.

### Why an Ordinary Test Can Mislead

Suppose we use K-means to form groups, then run a t-test comparing their means on one of the clustering features. We have used the same measurements twice: first to choose groups that differ, then to test that difference as though the groups had been specified independently.

This is **double dipping**, a problem for **post-selection inference**: the ordinary t-test does not account for how the groups were selected. It resembles leakage because information used to construct the result is reused to evaluate it. A small p-value can partly reflect that construction, even when the observations came from one population.

The t-test tests a difference in means; it does not test whether clusters are real populations. Statistical significance is not guaranteed after clustering, either. To see the problem in this example, generate 400 observations from one Gaussian population, ask K-means for two clusters, and apply an ordinary t-test to coordinate 1.

In [ ]:
single_cloud = np.random.default_rng(108).multivariate_normal(
    mean=[0, 0], cov=[[1, 0.4], [0.4, 1]], size=400,
)
single_fit = KMeans(n_clusters=2, n_init=20, random_state=7130).fit(single_cloud)
single_labels = single_fit.labels_
naive_result = ttest_ind(
    single_cloud[single_labels == 0, 0],
    single_cloud[single_labels == 1, 0],
    equal_var=False,
)
print(f"Ordinary post-clustering t-test p-value: {naive_result.pvalue:.2e}")
print("This p-value is invalid as evidence of separate populations after this selection.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharex=True, sharey=True,
                         layout="constrained")
axes[0].scatter(*single_cloud.T, color="0.4", s=20, alpha=0.7)
axes[0].set(title="Generated From One Gaussian Population", xlabel="Coordinate 1",
            ylabel="Coordinate 2")
axes[0].set_aspect("equal", adjustable="box")
plot_groups(axes[1], single_cloud, single_labels, "The Same Observations Split by K-Means")
plt.show()

The p-value is extremely small even though the observations came from one Gaussian population. K-means supplied the split using the same coordinates we tested. The ordinary p-value calculation ignores that selection, so its usual interpretation is unreliable here.

**A difference we deliberately created is not independent evidence that distinct populations exist.**

Unlike typical train/test leakage, a simple holdout does not necessarily fix this. Assigning new observations to clusters using these same features can also create differences by construction. Shuffling the final labels would likewise omit how we found the groups. The next comparison includes the clustering procedure itself: fit clusters to each simulated reference dataset before measuring separation.

### A Single-Population Reference

We ask whether one continuous Gaussian cloud could produce a split as strong as the one we found. We fit a single Gaussian to the data, generate many new datasets from it, and ask K-means to split each into two groups. We then compare how much those splits reduce squared distances with the reduction in our original data. If our split is unusually strong, that is evidence that the single-Gaussian description may be inadequate, but it does not prove there are exactly two real or useful groups.

The reference cloud can be elongated or tilted. Choosing a Gaussian is an assumption about what one population might look like.

Return to the two cloud examples: the single Gaussian used in the t-test demonstration and the overlapping clouds used for GMM. For each, compare K-means inertia with two centers to inertia with one center. Divide the two-center value by the one-center value. A ratio of 0.5 means that two centers leave half the original sum of squared distances. Smaller ratios mean a larger improvement from splitting the data.

In [ ]:
def two_center_ratio(x):
    two_centers = KMeans(n_clusters=2, n_init=20, random_state=7130).fit(x)
    total_dispersion = np.sum((x - x.mean(axis=0)) ** 2)
    return two_centers.inertia_ / total_dispersion


print(f"One-cloud ratio: {two_center_ratio(single_cloud):.3f}")
print(f"Overlapping-cloud ratio: {two_center_ratio(clouds):.3f}")

The ratios are about 0.535 for the single cloud and 0.424 for the overlapping clouds. The second split removes more of the original squared distance. For each dataset separately, we fit its own single-Gaussian reference and generate 499 datasets of the same size for comparison.

This is a **parametric bootstrap**: we generate new observations from a fitted model, whereas the ARI check sampled existing rows. Refitting K-means in every simulation includes the grouping step that the ordinary t-test ignored.

In each histogram below, the bars count simulated datasets, and the vertical line marks the observed ratio. A line far to the left means the observed split improved the fit more than most simulated splits.

In [ ]:
def gaussian_reference_comparison(x, repetitions=499, seed=806):
    fitted_mean = x.mean(axis=0)
    fitted_covariance = np.cov(x, rowvar=False, ddof=0)
    observed_ratio = two_center_ratio(x)
    reference_rng = np.random.default_rng(seed)
    reference_ratios = np.empty(repetitions)
    for repeat in range(repetitions):
        reference_sample = reference_rng.multivariate_normal(
            fitted_mean, fitted_covariance, size=len(x),
        )
        reference_ratios[repeat] = two_center_ratio(reference_sample)
    # Smaller ratios indicate greater separation by this statistic.
    at_least_as_separated = np.count_nonzero(reference_ratios <= observed_ratio)
    tail_estimate = (at_least_as_separated + 1) / (repetitions + 1)
    return observed_ratio, reference_ratios, at_least_as_separated, tail_estimate


reference_results = {
    "One Gaussian cloud": gaussian_reference_comparison(single_cloud),
    "Overlapping clouds": gaussian_reference_comparison(clouds),
}
reference_summary = pd.DataFrame([
    {"Data": name, "Observed ratio": result[0],
     "Reference samples at least as separated": result[2],
     "Approximate lower-tail estimate": result[3]}
    for name, result in reference_results.items()
]).set_index("Data")
display(reference_summary.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True,
                         layout="constrained")
all_ratios = np.concatenate([np.r_[r[0], r[1]] for r in reference_results.values()])
bins = np.linspace(all_ratios.min() - 0.01, all_ratios.max() + 0.01, 26)
for ax, (name, result) in zip(axes, reference_results.items(), strict=True):
    observed, simulated, count, tail = result
    ax.hist(simulated, bins=bins, color=COLORS[0], alpha=0.75,
            label="499 refitted Gaussian reference samples")
    ax.axvline(observed, color=COLORS[1], linewidth=2.5, label="Observed ratio")
    ax.set(title=name, xlabel="Two-center / one-center dispersion ratio",
           ylabel="Reference samples")
    ax.legend(fontsize=8, loc="upper left")
plt.show()

For the single cloud, about 45% of the simulations produce a ratio as small as or smaller than the observed one. Its split is ordinary under this reference, despite the tiny t-test p-value.

For the overlapping clouds, none of the 499 simulations produces a ratio as small as the observed one. The table reports 0.002 because we add one to both the count and denominator: $1/500$. This avoids reporting zero from a finite simulation. The observed split is unusually strong compared with splits of the fitted Gaussian samples.

This is approximate evidence against the chosen one-Gaussian description. We estimated that reference from the data, and have not checked whether a cutoff such as 0.05 would have the intended false-positive rate. A single population with a skewed or heavy-tailed shape could also depart from this reference; an unusual split alone does not establish multiple populations.

### Pause: What Has the Reference Comparison Established?

A colleague says, “None of the simulations split this strongly, so there must be exactly two real groups.” What did we compare, and what part of that claim remains untested?

##### Answer

We compared the observed split with splits of datasets generated from one fitted Gaussian. The result suggests that this reference does not explain the separation well. It does not choose among all possible population counts or establish what the groups mean. The reported 0.002 is an approximate simulation-tail estimate, not the probability that the one-population explanation is true. Check the reference assumption and seek other evidence about the groups.

### Principled Validation

Use a reference comparison when you have a clear question and a population model you can justify. It is not a routine checkbox. State the features, statistic, fitting procedure, and any choices made after seeing the results.

Suppose we tried many counts and feature sets, then reported only the strongest separation. Simulating that final fit alone would omit our search for a strong result. Either set the question before looking or include the relevant search in the reference procedure. A formal significance test also needs evidence that its false-positive rate is controlled.

Combine this evidence with fit, cluster count, sensitivity, stability, and usefulness. Each should answer a distinct question. Several scores based on the same geometry can agree without providing independent confirmation.

## What was the Question?

For the service team, differences in request count and duration describe the proposed groups because those measurements helped create them. Request content, future service demand, or another period's records could provide additional evidence about whether the profiles matter. A useful grouping might support a practical decision without proving that customers fall into discrete natural types.

Which grouping would you take to the service team, and why? Name the next piece of evidence you would seek before changing support. You may decide that request count and duration are more useful as continuous measurements than as categories.